# Prototype 1 

**Hypothesis 1**: US large-cap stocks that fall by at least 8% in one trading day exhibit a measurable tendency to subsequently recover. 

### Parameters: 

- **Event**: Daily close-to-close returns ≤ -8% 
- **Universe**: Large-cap US equities – for learning and prototyping purposes, a small basket of recognisable large US companies is used: 
    - AAPL (Apple Inc.)
    - MSFT (Microsoft Corporation)
    - AMZN (Amazon.com, Inc) 
    - GOOGL (Alphabet Inc., parent company of Google)
    - META (Meta Platforms, Inc.)
    - JPM (JPMorgan Chase & Co.)
    - JNJ (Johnson & Johnson) 
    - XOM (Exxon Mobil Corporation, energy sector & integrated oil and gas industry) 
    - WMT (Walmart Inc.)
    - NVDA (NVIDIA Corporation) 
    - SPY --> for modelling the wider market (SPDR S&P 500 ETF Trust - a basket of all 500 large-cap stocks in the S&P 500)
- **Initial Historical Period**: 10 years 
- **Outcome**: Whether the stock rises at least 5% from the post-drop close within the following 10 trading days. 
- **Secondary Variables**: General market direction and company profitability
- **Primary Metric**: Percentage of qualifying events that achieve the 5% recovery threshold within 10 trading days.  

## Import Libraries

Note for the future - It is worth doing an analysis on the different open-source financial data APIs that exist to see which is the most accurate and use that one. Also worth investigating the level of detail we need from the data for the stock-bot.

Yahoo! Finance is used as a starting point. 

In [1]:
# If yfinance is not installed, uncomment the next line:
!pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 15.3 MB/s  0:00:00

   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   ---------------- ----------------------- 2/5 [peewee]
   ---------------- ----------------------- 2/5 [peewee]
   ---------------- ----------------------- 2/5 [peewee]
   ---------------- ----------------------- 2/5 [peewee]
   ---------------- ----------------------- 2/5 [peewee]
   ------------------------ --------------- 3/5 [curl_cffi]
   ------------------------ --------------- 3/5 [curl_cffi]
   ------------------------ --------------- 3/5 [curl_cffi]
   -------------------------------- ------- 4/5 [yfinance]
   -------------------------------- ------- 4/5 [yfinance]
   --------------------------

## Download a Small Prototype Dataset

In [3]:
#10 large-cap US equities
tickers = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META",
    "JPM", "JNJ", "XOM", "WMT", "NVDA"
]

#10 years of historical data --> 01 Jan 2016 til 01-Jan-2026 
start_date = "2016-01-01"
end_date = "2026-01-01"

#datastruct to hold Yahoo! Finance stock data
prices = yf.download(
    tickers, #list of tickers to download as defined above
    start=start_date, #download start date given as string in format YYYY-MM-DD
    end=end_date, #download end date given as string in format YYYY-MM-DD
    auto_adjust=False, #Adjust all OHLC automatically --> smooth out the artificial effect of corporate events (i.e.,stock split, dividend) to reflect true investment performance? Default is True
    progress=True
)

prices.head()

[*********************100%***********************]  10 of 10 completed


Price       Adj Close                                                          \
Ticker           AAPL       AMZN      GOOGL        JNJ        JPM        META   
Date                                                                            
2016-01-04  23.688667  31.849501  37.638252  74.967827  48.184067  101.330162   
2016-01-05  23.095045  31.689501  37.741833  75.281181  48.267372  101.835732   
2016-01-06  22.643091  31.632500  37.632805  74.900673  47.570614  102.073631   
2016-01-07  21.687449  30.396999  36.724358  74.027740  45.646877   97.067604   
2016-01-08  21.802122  30.352501  36.224300  73.236862  44.624401   96.482735   

Price                                                  ...     Volume  \
Ticker           MSFT      NVDA        WMT        XOM  ...       AAPL   
Date                                                   ...              
2016-01-04  47.680958  0.788626  16.966858  49.002327  ...  270597600   
2016-01-05  47.898495  0.801295  17.369907  49.419865  ...  223164000   
2016-01-06  47.028389  0.768161  17.543831  49.008636  ...  273829600   
2016-01-07  45.392620  0.737708  17.952412  48.224201  ...  324377600   
2016-01-08  45.531837  0.721872  17.541073  47.249969  ...  283192000   

Price                                                                    \
Ticker           AMZN     GOOGL       JNJ       JPM      META      MSFT   
Date                                                                      
2016-01-04  186290000  67382000  12722800  25393200  37912400  53778000   
2016-01-05  116452000  45216000   6467200  16566700  23258200  34079700   
2016-01-06  106584000  48206000   7733800  22961500  25096200  39518900   
2016-01-07  141498000  63132000   9433100  27630900  45172900  56564900   
2016-01-08  110258000  47506000   9766700  22373300  35402300  48754000   

Price                                      
Ticker           NVDA       WMT       XOM  
Date                                       
2016-01-04  358076000  35967600  20400100  
2016-01-05  490272000  39978000  11993500  
2016-01-06  449344000  49693800  18826900  
2016-01-07  645304000  79290000  21263800  
2016-01-08  398472000  53303700  19033600  

[5 rows x 60 columns]

Downloaded data is in a multiIndex dataframe. The **adjusted closing price** for each stock (without distortions from corporate events) is extracted next, as it gives a cleaner time series to work with for return calculations. 

In [20]:
#A new table with adjusted closing prices of selected stocks
adj_close = prices["Adj Close"].copy()

adj_close.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,WMT,XOM
Date,,,,,,,,,,
2016-01-04,23.688667,31.849501,37.638252,74.967827,48.184067,101.330162,47.680958,0.788626,16.966858,49.002327
2016-01-05,23.095045,31.689501,37.741833,75.281181,48.267372,101.835732,47.898495,0.801295,17.369907,49.419865
2016-01-06,22.643091,31.632500,37.632805,74.900673,47.570614,102.073631,47.028389,0.768161,17.543831,49.008636
2016-01-07,21.687449,30.396999,36.724358,74.027740,45.646877,97.067604,45.392620,0.737708,17.952412,48.224201
2016-01-08,21.802122,30.352501,36.224300,73.236862,44.624401,96.482735,45.531837,0.721872,17.541073,47.249969


In [5]:
# Pandas.dataframe.pct_change() --> Computes the fractional change from the immediately previous row
daily_returns = adj_close.pct_change()

daily_returns.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,WMT,XOM
Date,,,,,,,,,,
2016-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-05,-0.025059,-0.005024,0.002752,0.004180,0.001729,0.004989,0.004562,0.016064,0.023755,0.008521
2016-01-06,-0.019569,-0.001799,-0.002889,-0.005054,-0.014435,0.002336,-0.018166,-0.041350,0.010013,-0.008321
2016-01-07,-0.042205,-0.039058,-0.024140,-0.011655,-0.040440,-0.049043,-0.034783,-0.039645,0.023289,-0.016006
2016-01-08,0.005288,-0.001464,-0.013617,-0.010684,-0.022400,-0.006025,0.003067,-0.021466,-0.022913,-0.020202


In [21]:
#sanity check to identify the worst-performing days for each of the selected stocks by pct_change
daily_returns.min().sort_values()

Ticker
META    -0.263901
NVDA    -0.187559
JPM     -0.149649
MSFT    -0.147390
AMZN    -0.140494
AAPL    -0.128647
XOM     -0.122248
GOOGL   -0.116341
WMT     -0.113757
JNJ     -0.100379
dtype: float64

Meta’s worst single trading day in the 2016–2026 dataset was about −26.4%.
Likewise NVDA had a worst day of about −18.8%, JPM about −15.0%, and so on. 

Importantly, every one of the ten stocks had at least one day worse than −8%. 

So, we continue the investigation!

## Defining the Rule

Identify events in data based on the rule daily return <= -8%


In [9]:
drop_threshold = -0.08 #-8%

large_drop_events = daily_returns <= drop_threshold

large_drop_events.head() #event detector table (consists of True/False for each day in the historical dataset)

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,WMT,XOM
Date,,,,,,,,,,
2016-01-04,False,False,False,False,False,False,False,False,False,False
2016-01-05,False,False,False,False,False,False,False,False,False,False
2016-01-06,False,False,False,False,False,False,False,False,False,False
2016-01-07,False,False,False,False,False,False,False,False,False,False
2016-01-08,False,False,False,False,False,False,False,False,False,False


In [10]:
#count how many events occurred for each company

event_counts = large_drop_events.sum().sort_values(ascending=False)

event_counts

Ticker
NVDA     19
META     10
AAPL      5
AMZN      5
JPM       5
XOM       5
GOOGL     4
WMT       4
MSFT      2
JNJ       1
dtype: int64

In [22]:
events = (
    daily_returns
    .stack()
    .reset_index()
)
#stack changes daily_returns to a long-format table. Previously it was date as the rows, ticker as the columns, and return as the values in the table
# Date       Ticker    Return
# Jan 1      AAPL       0.01
# Jan 1      MSFT      -0.02
# Jan 1      META       0.03
# ... 

events.columns = ["Date", "Ticker", "Daily_Return"] #standardized column names 

#Events are occurrences where the defined threshold was met. In this case a drop less than or equal to -8%
events = events[
    events["Daily_Return"] <= drop_threshold
].copy()

#Sort the long-format events table by Date
events = events.sort_values("Date").reset_index(drop=True)

#show head of table
events.head(10)

,Date,Ticker,Daily_Return
0,2017-02-23,NVDA,-0.092723
1,2018-02-05,NVDA,-0.084875
2,2018-02-20,WMT,-0.101833
3,2018-07-26,META,-0.189609
4,2018-10-24,NVDA,-0.097937
5,2018-11-16,NVDA,-0.187559
6,2018-11-19,NVDA,-0.119990
7,2018-12-14,JNJ,-0.100379
8,2019-01-03,AAPL,-0.099607
9,2019-01-28,NVDA,-0.138245


In [13]:
print(f"Total large-drop events: {len(events)}") #across the set of 10 large-cap US companies

events.sort_values("Daily_Return").head(10) #sorted with most negative daily returns showing at the top 

Total large-drop events: 60


,Date,Ticker,Daily_Return
35,2022-02-03,META,-0.263901
44,2022-10-27,META,-0.245571
3,2018-07-26,META,-0.189609
5,2018-11-16,NVDA,-0.187559
25,2020-03-16,NVDA,-0.184521
52,2025-01-27,NVDA,-0.169682
22,2020-03-16,JPM,-0.149649
24,2020-03-16,MSFT,-0.147390
23,2020-03-16,META,-0.142530
36,2022-04-29,AMZN,-0.140494


## Bringing in Forward Return

In [23]:
# Add the adjusted closing price on the event date
#Go through the events dataframe one row at a time. 
#For each row, use its Date and Ticker to look up the corresponding adjusted closing price in adj_close, and store that in a new 'Event_Price' column
#axis = 1 tells .apply() to operate row by row 
events["Event_Price"] = events.apply(
    lambda row: adj_close.loc[row["Date"], row["Ticker"]], #lambda is a tiny anonymous function.
    axis=1
)

events.head()

#alternate formatting for lambda row: adj_close.loc[row["Date"], row["Ticker"]]
#def get_event_price(row):
#    return adj_close.loc[row["Date"], row["Ticker"]]

,Date,Ticker,Daily_Return,Event_Price
0,2017-02-23,NVDA,-0.092723,2.474879
1,2018-02-05,NVDA,-0.084875,5.276873
2,2018-02-20,WMT,-0.101833,27.425871
3,2018-07-26,META,-0.189609,174.725616
4,2018-10-24,NVDA,-0.097937,4.932814


In [24]:
# Function to examine the next 10 trading days after each event
#recovery threshold - 5% 
def check_recovery(row, recovery_threshold=0.05, forward_days=10):
    ticker = row["Ticker"]
    event_date = row["Date"]
    event_price = row["Event_Price"] #adjusted closing price

    # Find the location of the event date
    event_index = adj_close.index.get_loc(event_date)

    # Select the NEXT 10 trading days --> iloc: select data using integer positions, loc: for labels/names
    future_prices = adj_close[ticker].iloc[
        event_index + 1 : event_index + 1 + forward_days
    ]

    # If we don't have enough future data, return missing values
    if len(future_prices) == 0:
        return pd.Series([np.nan, np.nan])

    max_future_price = future_prices.max() #maximum future price over the next 10 days

    #maximum recovery is % change between event price and maximum future price
    max_recovery = (
        max_future_price - event_price
    ) / event_price

    recovered = max_recovery >= recovery_threshold

    return pd.Series([max_recovery, recovered])

In [17]:
#two new columns are added to events, one which shows the maximum % change in future price, second whether it meets the 5% criteria
events[
    ["Max_Recovery_10D", "Recovered_5pct_10D"]
] = events.apply(check_recovery, axis=1)

events.head(10)

,Date,Ticker,Daily_Return,Event_Price,Max_Recovery_10D,Recovered_5pct_10D
0,2017-02-23,NVDA,-0.092723,2.474879,0.039009,False
1,2018-02-05,NVDA,-0.084875,5.276873,0.165559,True
2,2018-02-20,WMT,-0.101833,27.425871,-0.010520,False
3,2018-07-26,META,-0.189609,174.725616,0.053501,True
4,2018-10-24,NVDA,-0.097937,4.932814,0.093777,True
5,2018-11-16,NVDA,-0.187559,4.067513,0.035152,False
6,2018-11-19,NVDA,-0.119990,3.579451,0.176296,True
7,2018-12-14,JNJ,-0.100379,107.526054,-0.019399,False
8,2019-01-03,AAPL,-0.099607,33.707916,0.096139,True
9,2019-01-28,NVDA,-0.138245,3.417376,0.108615,True


Note to self in the future. Consider using daily High price when building the bot. Right now for simplification the adjusted closing price of the day is used. This does not necessarily give you the highest price of the day.

## Headline numbers

In [18]:
valid_events = events.dropna(subset=["Recovered_5pct_10D"]) #drop any NULLS from dataset

hit_rate = valid_events["Recovered_5pct_10D"].mean() #whenever recovery >= 5%, hit rate

print(f"Number of events: {len(valid_events)}")
print(f"5% recovery hit rate: {hit_rate:.1%}")

Number of events: 60
5% recovery hit rate: 58.3%


In [19]:
#breakdown by company too 
recovery_by_stock = (
    valid_events
    .groupby("Ticker")
    .agg(
        Events=("Recovered_5pct_10D", "count"), #all events per company, counts True and False
        Successful_Recoveries=("Recovered_5pct_10D", "sum"), #only True events for company are summed
        Hit_Rate=("Recovered_5pct_10D", "mean") #mean = sum/count 
    )
    .sort_values("Hit_Rate", ascending=False)
)

recovery_by_stock

,Events,Successful_Recoveries,Hit_Rate
Ticker,,,
MSFT,2,2,1.000000
JPM,5,4,0.800000
META,10,7,0.700000
AAPL,5,3,0.600000
AMZN,5,3,0.600000
NVDA,19,11,0.578947
GOOGL,4,2,0.500000
XOM,5,2,0.400000
WMT,4,1,0.250000


## Preliminary Results

The prototype dataset identified 60 single-day decline events of 8% or greater across the ten-stock sample between 2016 and 2025. Of these events, 58.3% reached a closing price at least 5% above the event-day close within the subsequent ten trading days.

Recovery rates varied considerably between stocks, although the number of observations for individual companies remains too small to support stock-level conclusions. The distribution of events was also uneven, with more volatile equities accounting for a disproportionate share of qualifying declines.

These results are therefore treated as exploratory rather than evidence of a profitable mean-reversion strategy. The next stage will compare post-decline recovery rates against an unconditional baseline and investigate the magnitude and distribution of subsequent returns.

## Questions Left to Ponder 
Q1. What is the best finance API to use to get the most accurate real-time data? 

Q2. What happens when we use daily high price instead of adjusted closing price at end of day (simplification)? 

Q3. Hit rate is not the same thing as profitability! If a strategy traded every qualifying event, what would the distribution of gains and losses look like, and would the strategy make money overall?

Q4. Concepts of average return, median return, loss size, risk, expected return. 

Q5. How to deal with dominating behaviour effects of unusually volatile stocks? 

Q6. Should the threshold be an absolute −8% for every stock, or should “large decline” be defined relative to that stock's normal volatility?

Q7. What should the entry mechanics be for the simulated trading strategy (the final closing price is not known until EOD and cannot be the main metric on which trading is done)? 